# ಪಾಠ 05 - ಏಜೆಂಟಿಕ್ RAG


## ಸೆಟ್‌ಅಪ್

ಈ ನೋಟ್‌ಬುಕಿನಲ್ಲಿ ಮೈಕ್ರೋಸಾಫ್ಟ್ ಏಜೆಂಟ್ ಫ್ರೇಮ್‌ವರ್ಕ್ ಬಳಸಿ ಏಜೆಂಟಿಕ್ RAG (ರಿಟ್ರೈವಲ್-ಆಗ್ಮೆಂಟೆಡ್ ಜನರೇಶನ್) ಮಾದರಿಯನ್ನು ಪ್ರದರ್ಶಿಸಲಾಗುತ್ತದೆ.

**ಆವಶ್ಯಕತೆಗಳು:**
- `AZURE_SEARCH_SERVICE_ENDPOINT` — ನಿಮ್ಮ ಏಜ್ಯೂರ್ AI ಶೋಧನೆ ಸೇವೆ ಅಂತಿಮ ಬಿಂದುವು
- `AZURE_SEARCH_API_KEY` — ನಿಮ್ಮ ಏಜ್ಯೂರ್ AI ಶೋಧನೆ API ಕೀ
- ವಾತಾವರಣದ ಚರಗಳು ಮೂಲಕ ಸಂರಚಿಸಿದ ಏಜ್ಯೂರ್ ಓಪನ್‌ಎಐ ನಿಯೋಜನೆ
- ಏಜ್ಯೂರ್ CLI ಪ್ರಾಮಾಣೀಕರಿಸಲಾಗಿದೆ (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## ಏಜೆಂಟಿಕ್ RAG ಎಂದರೆ ಏನು?

ಸಾಂಪ್ರದಾಯಿಕ RAG ಒಂದು ನಿಶ್ಚಿತ ಪರಿಕ್ರಮೆಯನ್ನು ಅನುಸರಿಸುತ್ತದೆ: ದಾಖಲೆಗಳನ್ನು ಪಡೆದು, ನಂತರ ಉತ್ತರವನ್ನು ರಚಿಸುವುದು. **ಏಜೆಂಟಿಕ್ RAG** ಇದನ್ನು ಮೀರಿ ಏಜೆಂಟ್ಗೆ ಸ್ವಾಯತ್ತತೆಯನ್ನು ನೀಡುತ್ತದೆ ಯಾವಾಗ ಮತ್ತು ಹೇಗೆ ಮಾಹಿತಿಯನ್ನು ಪಡೆದುಕೊಳ್ಳಬೇಕೆಂದು ನಿರ್ಧರಿಸುವ ಮಗುವನ್ನು.

ಏಜೆಂಟಿಕ್ RAG ಮೂಲಕ, ಏಜೆಂಟ್ ಈ ಕಾರ್ಯಗಳನ್ನು ಮಾಡಬಹುದು:
- ಪ್ರಶ್ನೆಗೆ ಉತ್ತರಿಸುವ ಮೊದಲು retrieval ಅಗತ್ಯವಿದೆಯೇ ಎಂದು **ನಿರ್ಧರಿಸಬಹುದು**
- ಯಾವ ಡೇಟಾ ಮೂಲ ಅಥವಾ ಸಾಧನವನ್ನು ಕೇಳಬೇಕೆಂದು **ఎంపిక చేయగలడు**
- ಪಡೆದ ಫಲಿತಾಂಶಗಳನ್ನು **ಮೌಲ್ಯಮಾಪನ ಮಾಡಿ**, ಮೊದಲು ಪ್ರಯತ್ನವು ಸಾಕಾಗದಿದ್ದರೆ ಅನಂತರ retrievalಗಳನ್ನು ನಡೆಸಬಹುದು
- ಹಲವಾರು retrieval ಹಂತಗಳಿಂದ ಮಾಹಿತಿ **ಸಂಯೋಜಿಸಿ** ಸಮಗ್ರ ಉತ್ತರವನ್ನು ರಚಿಸಬಹುದು

ಇದು ಏಜೆಂಟ್ನನ್ನು ಸ್ಥಿರ retrieval-ನಂತರ-ಉತ್ಪಾದನೆ ಪರಿಕ್ರಮೆಯಿಗಿಂತ ಹೆಚ್ಚು ನಿಲುಕುವ ಹಾಗೂ ನಿಖರವಾಗುವಂತೆ ಮಾಡುತ್ತದೆ.


## ಶೋಧಕಾರಿ ಉಪಕರಣವನ್ನು ರಚಿಸುವುದು

Agentic RAG ನಲ್ಲಿ, ಹೊರಗಿನ ದತ್ತಾಂಶ ಮೂಲಗಳನ್ನು ಏಜೆಂಟ್ ಬೇಡಿಕೆಯಂತೆ ಕರೆಸಿಕೊಳ್ಳಬಹುದಾದ **ಉಪಕರಣಗಳಾಗಿ** обಿಸುತ್ತಾರೆ. ಇದರಿಂದ ಏಜೆಂಟ್ ಅನ್ನು ಹಿಂಪಡೆಯುವಿಕೆ ಇನ್ನೊಂದು ಕ್ರಮವಂತೆ ಮಾತ್ರ ನೋಡಿಕೊಳ್ಳಬಹುದು, ಬದಲು ಅದು ಒಂದು ಅಗತ್ಯದ ಹೆಜ್ಜೆ ಅಲ್ಲ.

ಕೆಳಗೆ ನಾವು ಒಂದು ಪ್ರಯಾಣ ಜ್ಞಾನ ಆಧಾರವನ್ನು ಸಂರಚಿಸಿ, ಏಜೆಂಟ್ ಗಮ್ಯತೆಯ ಮಾಹಿತಿಯನ್ನು ಹುಡುಕಲು ಕರೆಸಿಕೊಳ್ಳಬಹುದಾದ ಉಪಕರಣದಂತೆ ಅದನ್ನು ಪ್ರದರ್ಶಿಸುತ್ತೇವೆ.


In [ ]:
TRAVEL_KNOWLEDGE_BASE = {
    "Barcelona": "Barcelona is Spain's cosmopolitan capital of Catalonia. Best visited Mar-May or Sep-Nov. Known for Gaudí architecture, La Rambla, beaches. Average daily cost: $150-200.",
    "Tokyo": "Tokyo is Japan's capital, mixing ultramodern with traditional. Best visited Mar-Apr (cherry blossoms) or Oct-Nov. Known for Shibuya, temples, sushi. Average daily cost: $200-250.",
    "Paris": "Paris is France's capital and a global center for art, fashion, and culture. Best visited Apr-Jun or Sep-Oct. Known for Eiffel Tower, Louvre, cuisine. Average daily cost: $180-250.",
    "Cape Town": "Cape Town sits on South Africa's southwest tip. Best visited Nov-Mar. Known for Table Mountain, wine regions, wildlife. Average daily cost: $100-150.",
}


@tool(approval_mode="never_require")
def search_travel_knowledge(
    query: Annotated[str, "The search query about a travel destination"]
) -> str:
    """Search the travel knowledge base for destination information."""
    results = []
    for destination, info in TRAVEL_KNOWLEDGE_BASE.items():
        if query.lower() in destination.lower() or any(
            word in info.lower() for word in query.lower().split()
        ):
            results.append(f"**{destination}**: {info}")
    return (
        "\n\n".join(results)
        if results
        else "No matching destinations found in the knowledge base."
    )

## RAG ಏಜೆಂಟ್ ನಿರ್ಮಾಣ

ಈಗ ನಾವು ಯಾವಾಗಲೂ ಉತ್ತರಿಸುವ ಮುಂಚೆ তথ্যವನ್ನು ಪಡೆದುಕೊಳ್ಳಲು ಸೂಚಿಸಲಾದ ಏಜೆಂಟ್ ಅನ್ನು ರಚಿಸುತ್ತೇವೆ. ಈ ಏಜೆಂಟ್ ತನ್ನ ಉತ್ತರಗಳನ್ನು ತನ್ನ ತರಬೇತಿ ಡೇಟಾದ ಮೇಲೆ ಅವಲಂಬಿಸಿಕೊಂಡು ಬದಲಿಗೆ ಜ್ಞಾನ ಆಧಾರದ ಮೇಲೆ ನೆಲೆಗೊಳಿಸಲು `search_travel_knowledge` ಯಂತ್ರವನ್ನು ಬಳಸುತ್ತದೆ.


In [ ]:
agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGAgent",
    instructions="""You are a knowledgeable travel advisor. Before answering questions about destinations:
1. ALWAYS search the travel knowledge base first
2. Base your answers on retrieved information
3. If information is not in the knowledge base, say so clearly
4. Provide specific details like costs, best seasons, and highlights.""",
)

response = await agent.run(
    "I'm interested in visiting somewhere with great architecture. What destinations would you recommend?",
    )
print(response)

## ಇಟರೇಟಿವ್ ರಿಟ್ರೀವಲ್ — ದ ಮೇಕರ್-ಚೆಕ್‍ರ್ ಮಾದರಿ

ಏಜೆಂಟಿಕ್ RAG ನ ಒಂದು ಪ್ರಮುಖ ಪ್ರಯೋಜನವು **ಇಟರೇಟಿವ್ ರಿಟ್ರೀವಲ್** ಆಗಿದೆ. ಏಜೆಂಟ್ ಅದಿನ್ನ ಪ್ರಾಥಮಿಕ ಕಂಡುಹಿಡಿತಗಳನ್ನು ಪರಿಶೀಲಿಸಲು, ಪರಿಷ್ಕರಿಸಲು ಅಥವಾ ವಿಸ್ತರಿಸಲು ಅನೇಕ ಸುತ್ತುಗಳ ಹುಡುಕಾಟವನ್ನು ನಡೆಸಬಹುದು — "ಮೇಕರ್-ಚೆಕ್‍ರ್" ಕಾರ್ಯಪ್ರವಾಹದಂತೆ:

1. **ಮೇಕರ್ ಹಂತ**: ಏಜೆಂಟ್ ಪ್ರಾಥಮಿಕ ಮಾಹಿತಿಯನ್ನು ಪಡೆಯುತ್ತದೆ ಮತ್ತು ಉತ್ತರವನ್ನು ರಚಿಸುತ್ತದೆ.
2. **ಚೆಕ್‍ರ್ ಹಂತ**: ಏಜೆಂಟ್ ವಿವರಗಳನ್ನು ಪರಿಶೀಲಿಸಲು ಅಥವಾ ಖಾಲිಗಳನ್ನು ಭರ್ತಿ ಮಾಡಲು ಹೆಚ್ಚುವರಿ ರಿಟ್ರೀವಲ್‍ಗಳನ್ನು ಕಾರ್ಯಗತಗೊಳಿಸುತ್ತಾನೆ.

ಕೆಳಗಿನಂತೆ, ಏಜೆಂಟ್‌ಗೆ ಹಲವಾರು ಗಮ್ಯಸ್ಥಾನೆಗಳನ್ನು ಹೋಲಿಸುವ ಪ್ರಶ್ನೆ ಕೇಳಲಾಗಿದೆ, ಇದರಿಂದ ಅದು ಹಲವು ಸಾರಿ ಹುಡುಕಲು ಪ್ರೇರೇಪಿತವಾಗುತ್ತದೆ.


In [ ]:
checker_agent = await provider.create_agent(
    tools=[search_travel_knowledge],
    name="TravelRAGCheckerAgent",
    instructions="""You are a meticulous travel advisor who double-checks recommendations.
When answering travel questions:
1. Search for relevant destinations first
2. For each destination found, search again with the destination name to get full details
3. Compare the options using verified information
4. Present a final recommendation with specific costs, best travel times, and highlights
5. If any detail seems incomplete, search once more to confirm before responding.""",
)

response = await checker_agent.run(
    "I have a $175/day budget and want to travel in April. Which destinations fit my budget and timing?",
    )
print(response)

## ಸಾರಾಂಶ

ಈ ಪಾಠದಲ್ಲಿ ನೀವು Microsoft Agent Framework ಬಳಸಿ **Agentic RAG** ವ್ಯವಸ್ಥೆಯನ್ನು ಹೇಗೆ ನಿರ್ಮಿಸಬೇಕೆಂದು ಕಲಿತಿರಿ:

- **Agentic RAG** ಏಜೆಂಟ್ಗಳು ಸ್ವತಃ ಮಾಹಿತಿ ಎತ್ತಿಕೊಳ್ಳುವ ಸಮಯವನ್ನು ನಿರ್ಧರಿಸಲು ಅನುಮತಿಸುತ್ತದೆ, ಇದರಿಂದ ಮಾಹಿತಿ ತೆಗೆದುಕೊಳ್ಳುವುದು ನಿಶ್ಚಿತವಾಗಿರುವುದಾಗಿಂದ ಗತಿಯುತವಾಗಿರುತ್ತದೆ.
- **ಪರಿಣಾಮಗಳನ್ನು ಡೇಟಾ ಮೂಲಗಳಾಗಿ ಬಳಸುವುದು**: ಬಾಹ್ಯ ಜ್ಞಾನ ಭಂಡಾರಗಳು (ಉದಾ: Azure AI Search) ಏಜೆಂಟ್ ಬಳಸಬಹುದಾದ ಸಾಧನಗಳಾಗಿ包ಲ್ಪಡುತ್ತವೆ.
- **ಪುನರಾವರ್ತಿತ ತಗಲಿಕೆ**: ಮಾಕರ್-ಚೆಕಾರ್ ಮಾದರಿ ಏಜೆಂಟ್ಗೆ ಹಲವಾರು ಸಂಗ್ರಹಣಾ ಸುತ್ತುಗಳನ್ನು ನೆರವೇರಿಸಲು ಸಹಾಯ ಮಾಡುತ್ತದೆ — ಹುಡುಕಾಟ, ದೃಢೀಕರಣ ಮತ್ತು ಪರಿಶೋಧನೆ — ಅಂತಿಮ ಉತ್ತರ ನೀಡುವುದಕ್ಕಿಂತ ಮುಂಚೆ.

ಉತ್ಪಾದನೆಯಲ್ಲಿ, ನೀವು ಸ್ಮೃತಿ ಆಧಾರಿತ `TRAVEL_KNOWLEDGE_BASE` ಅನ್ನು ಬೃಹತ್ ಪ್ರಮಾಣದ ಪ್ರಯಾಣ ದಾಖಲೆ ಸಂಗ್ರಹಣೆ ನಿರ್ವಹಿಸಲು ನಿಜವಾದ Azure AI Search ಸೂಚ್ಯಂಕದಿಂದ ಬದಲಾಯಿಸಬಹುದು.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ಮುಕ್ತಾಯ ಸೂಚನೆ**:  
ಈ ದಾಖಲೆ AI ಅನುವಾದ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ನ ಮೂಲಕ ಅನುವಾದಿಸಲಾಗಿದೆ. ನಾವು ಶುದ್ಧತೆಗಾಗಿ ಪ್ರಯತ್ನಿಸಿದರೂ, ಸ್ವಯಂಕ್ರಿಯ ಅನುವಾದಗಳಲ್ಲಿ ತಪ್ಪುಗಳು ಅಥವಾ ಅಸತ್ಯತೆಗಳು ಇರಬಹುದೆಂದು ದಯವಿಟ್ಟು ಗಮನಿಸಿ. ಮೂಲ ಭಾಷೆಯಲ್ಲಿ ಇರುವ ದಾಖಲೆ ಆಧಾರಣೀಯವಾದ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಮಹತ್ವದ ಮಾಹಿತಿಗಾಗಿ ವೃತ್ತಿಪರ ಮಾನವ ಅನುವಾದವನ್ನು ಸೂಚಿಸಲಾಗುತ್ತದೆ. ಈ ಅನುವಾದವನ್ನು ಬಳಸುವುದರಿಂದ ಹುಟ್ಟುವ ಯಾವುದೇ ತಪ್ಪು ಅರ್ಥಮಾಡಿಕೊಳ್ಳುವಿಕೆ ಅಥವಾ ವಿವರಣೆಗಾಗಿ ನಾವು ಹೊಣೆಗಾರರಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
